In [ ]:
import pandas as pd
import networkx as nx
from sklearn.model_selection import GroupShuffleSplit

# 1. Load your dataset
df = pd.read_csv("../preprocessing/train_data_with_preprocessing2.csv")

# 2. Build the Bipartite Graph
G = nx.Graph()

# Add prefixes to avoid collisions between identical context and reply strings
df['ctx_node'] = "CTX_" + df['prev_clean'].astype(str)
df['rep_node'] = "REP_" + df['target_clean'].astype(str)

# Create edges based on rows
edges = list(zip(df['ctx_node'], df['rep_node']))
G.add_edges_from(edges)

# 3. Find Connected Components
components = list(nx.connected_components(G))

# Map every node to a unique Component ID
node_to_component = {}
for component_id, nodes in enumerate(components):
    for node in nodes:
        node_to_component[node] = component_id

# 4. Map the Component IDs back to the original dataframe
# (You only need to check the context node, as the reply node is guaranteed to have the same ID)
df['component_id'] = df['ctx_node'].map(node_to_component)

# 5. Split the data ensuring components are kept together
# GroupShuffleSplit ensures all rows with the same component_id stay in the same split
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)

train_idx, val_idx = next(gss.split(df, groups=df['component_id']))

train_df = df.iloc[train_idx].drop(columns=['ctx_node', 'rep_node', 'component_id', 'split', 'group', 'is_toxic', 'label_id', 'previous_reply', 'target_reply'])
val_df = df.iloc[val_idx].drop(columns=['ctx_node', 'rep_node', 'component_id', 'split', 'group', 'is_toxic', 'label_id', 'previous_reply', 'target_reply'])

print(f"Training rows: {len(train_df)}")
print(f"Validation rows: {len(val_df)}")

In [ ]:
train_df.rename(columns={'prev_clean': 'context', 'target_clean': 'reply'}, inplace=True)
val_df.rename(columns={'prev_clean': 'context', 'target_clean': 'reply'}, inplace=True)

from scipy.sparse import hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
  accuracy_score,
  f1_score,
  classification_report,
  confusion_matrix
)

In [ ]:
target_col = 'label'

In [ ]:
# Extract text columns for TF-IDF
X_context_train = train_df['context'].fillna("").astype(str)
X_reply_train = train_df['reply'].fillna("").astype(str)
y_train = train_df[target_col].fillna("nan").astype(str)

X_context_dev = val_df['context'].fillna("").astype(str)
X_reply_dev = val_df['reply'].fillna("").astype(str)
y_dev = val_df[target_col].fillna("nan").astype(str)

In [ ]:
# Word-level TF-IDF
context_word = TfidfVectorizer(
  analyzer="word",
  ngram_range=(1, 2),
  min_df=2,
  max_features=50000,
  sublinear_tf=True
)

reply_word = TfidfVectorizer(
  analyzer="word",
  ngram_range=(1, 2),
  min_df=2,
  max_features=50000,
  sublinear_tf=True
)

In [ ]:
# Character-level TF-IDF
context_char = TfidfVectorizer(
  analyzer="char_wb",
  ngram_range=(3, 5),
  min_df=2,
  max_features=50000,
  sublinear_tf=True
)

reply_char = TfidfVectorizer(
  analyzer="char_wb",
  ngram_range=(3, 5),
  min_df=2,
  max_features=50000,
  sublinear_tf=True
)

In [ ]:
# Fit only on training data, transform both
Cw_train = context_word.fit_transform(X_context_train)
Cw_dev = context_word.transform(X_context_dev)

Rw_train = reply_word.fit_transform(X_reply_train)
Rw_dev = reply_word.transform(X_reply_dev)

Cc_train = context_char.fit_transform(X_context_train)
Cc_dev = context_char.transform(X_context_dev)

Rc_train = reply_char.fit_transform(X_reply_train)
Rc_dev = reply_char.transform(X_reply_dev)

In [ ]:
# Combine all four sparse feature matrices
X_train = hstack([Cw_train, Cc_train, Rw_train, Rc_train], format="csr")
X_dev = hstack([Cw_dev, Cc_dev, Rw_dev, Rc_dev], format="csr")

print("Training feature matrix:", X_train.shape)
print("Development feature matrix:", X_dev.shape)